# Phase 12.3 — Search Evaluation Metrics

Load the Phase 12.2 benchmark, attach synthetic ranked IDs, and compute per-query and macro metrics.
No search execution.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.retrieval.evaluation.search_evaluation import (
    SearchEvaluationVariant,
    SearchMetricConfiguration,
    SearchRankedResultsForQuery,
    compute_search_query_metrics,
    default_search_benchmark_path,
    evaluate_search_variant_metrics,
    load_search_evaluation_benchmark,
)

In [ ]:
benchmark = load_search_evaluation_benchmark(default_search_benchmark_path(ROOT))
first_query = sorted(benchmark.queries, key=lambda row: row.query_id)[0]
synthetic_ranking = tuple(j.product_id for j in first_query.relevance_judgments)
config = SearchMetricConfiguration(k_values=(1, 5, 10), evaluation_top_k=10)
row = compute_search_query_metrics(first_query, synthetic_ranking, config)
print("query:", row.query_id)
print("precision@5:", row.precision_at_k[5])
print("recall@5:", row.recall_at_k[5])
print("mrr:", row.reciprocal_rank)

In [ ]:
ranked = tuple(
    SearchRankedResultsForQuery(
        query_id=query.query_id,
        ranked_product_ids=tuple(j.product_id for j in query.relevance_judgments),
    )
    for query in benchmark.queries
)
variant = SearchEvaluationVariant(
    variant_name="notebook_synthetic_perfect_recall_prefix",
    variant_version="0.0.0",
)
result = evaluate_search_variant_metrics(benchmark, variant, ranked, metric_configuration=config)
print("aggregate mrr:", result.aggregate.mrr)
print("mean precision@5:", result.aggregate.mean_precision_at_k[5])